# 1 — The Perceptron algorithm

Within this notebook we'll:

1. Recap the **Perceptron Learning Algorithm**.
2. Implement the simple learning rule to train a single Perceptron

Nice-to-have: undertsand the "./0-torch-beginners.ipynb" lecture

## Recap - Perceptron learning rule

We are given a dataset $\mathcal{D} = \{(x^{(i)}, y^{(i)})\}_{i=1}^N$ of $N$ labelled examples, where each input $x^{(i)} \in \mathbb{R}^d$ is a feature vector and each label $y^{(i)} \in \{-1, +1\}$ marks its class. The task is **binary classification**: given a new $x$, predict its label $y$.

**Geometric picture.** A linear classifier draws a hyperplane $w^\top x = 0$ through feature space and assigns the label by which side of that hyperplane a point falls on. Rosenblatt's goal is to find a normal vector $w^* \in \mathbb{R}^d$ such that every training point sits on the *correct* side:

$$
    \begin{cases}
        w^{*\top}x^{(i)} > 0, & \text{if}\ y^{(i)}=+1 \\
        w^{*\top}x^{(i)} < 0, & \text{if}\ y^{(i)}=-1
    \end{cases}
$$

Both cases are captured by the single compact condition

$$
    y^{(i)}\, w^{*\top} x^{(i)} > 0 \quad \forall\, i \in [N].
$$

This is why we use $y \in \{-1, +1\}$ rather than $\{0, 1\}$: the product $y \cdot w^\top x$ is positive **iff** $\operatorname{sgn}(w^\top x)$ agrees with $y$, so a single scalar tells us whether a sample is classified correctly.

**The perceptron function.** A single-neuron perceptron with weights $w$ predicts

$$f_w(x^{(i)}) = \operatorname{sgn}\!\left(w^\top x^{(i)}\right) = \operatorname{sgn}\!\left(\sum_{j=1}^d w_j\, x^{(i)}_j\right) = \hat{y}^{(i)},$$

where

$$
    \operatorname{sgn}(x) =
    \begin{cases}
      +1, & \text{if}\ x>0 \\
      0, & \text{if}\ x=0 \\
      -1, & \text{if}\ x<0
    \end{cases}
$$

So the goal "$y^{(i)}\, w^{*\top} x^{(i)} > 0$ for all $i$" is just "$f_{w^*}$ classifies every training point correctly". The hyperplane $w^\top x = 0$ always passes through the origin.

Given these preliminaries, the *Rosenblatt's training algorithm* is the following:

1. Pick a sample of index $i \in [N]$ and compute $\hat{y}^{(i)} = f_{w_t}(x^{(i)})$ using the current set of weights $w_t$
2. If $\hat{y}^{(i)} = y^{(i)}$, then $i$ is correctly classified and there's nothing we can do
3. Otherwise, we update the weights as follows:
    $$
        w_{t+1} = w_t + y^{(i)}x^{(i)}
    $$
4. Iterate until a specified number of epochs as passed or until no sample is misclassified

This algorithm needs an important assumption: the data we are classifying must be **linearly separable**, i.e., there exists a vector $w^*$ that separates the data with some positive margin $\gamma > 0$:

$$
    y^{(i)}w^{*\top}x^{(i)} \geq \gamma \quad \forall i \in [N]
$$

## Adding the bias and the convergence bound

To learn a *shifted* hyperplane $w^\top x + b = 0$ we need to augment every input with a constant coordinate $x^{(i)} \leftarrow (x^{(i)}, 1)$ and let the last entry of $w$ absorb the bias $b$, i.e.:

- $\tilde{w}^* = (w^* \quad b)^\top, \quad b \in \mathbb{R}$
- $\tilde{x}^{(i)} = (x^{(i)} \quad 1)^\top$

then, $\tilde{w}^{*\top}\tilde{x}^{(i)} = w^{*\top}x^{(i)} + b$

From Stefano's lectures, to prove the convergence of the *Perceptron Learning Rule* we needed the following assumptions:

1. there exists a vector $w^*$ that separates the data with some positive margin $\gamma > 0$, i.e., ${y^{(i)}\, w^{*\top} x^{(i)} > \gamma \quad \forall\, i \in [N]}$
2. $\lVert x^{(i)} \rVert \leq R  \quad \forall i \in [N]$
3. $\lVert w^* \rVert = 1$

We need to update this assumptions with our augmented weights $\tilde{w}$ and inputs $\tilde{x}^{(i)}$:

1. there exists a vector $\tilde{w}^*$ that separates the data with some positive margin $\gamma > 0$, i.e., ${y^{(i)}\, \tilde{w}^{*\top} \tilde{x}^{(i)} = y^{(i)}(w^{*\top}x^{(i)} + b) > \gamma \quad \forall\, i \in [N]}$
2. $\lVert \tilde{x}^{(i)} \rVert \leq M  \quad \forall i \in [N]$, which means that ${\lVert \tilde{x}^{(i)} \rVert^2 = \lVert x^{(i)} \rVert^2 + 1 \leq R^2 + 1 \implies \lVert \tilde{x}^{(i)} \rVert \leq \sqrt{R^2 + 1}  \quad \forall i \in [N]}$
3. $\lVert \tilde{w} \rVert = 1$

Then, recalling again Stefano's lectures, the number of misclassified steps are upper-bounded by:

$$
    \tau \leq \left(\frac{\sqrt{R^2 + 1}}{\gamma}\right)^2
$$

## Creating the dataset

In [ ]:
# Imports

import matplotlib.pyplot as plt
import numpy as np
import torch

## Idea behind data generation

Let $\hat{w} = w / \lVert w \rVert$, then every $x \in \mathbb{R}^d$ can be decomposed as ${x = \underbrace{(\hat{w}^\top x)\,\hat{w}}_{\text{projection onto } \hat{w}} + \underbrace{\bigl(x - (\hat{w}^\top x)\,\hat{w}\bigr)}_{\text{projection onto } w^\perp}}$. If we call $\alpha = \hat{w}^\top x$ and the second term just $x^\perp$, then $x = \alpha \hat{w} + x^\perp$ and the constraint

$$y\,(w^\top x + b) \;\geq\; \gamma \qquad (\gamma \geq 0)$$

collapses to a one-dimensional condition on $\alpha$. Plugging in $x = \alpha \hat{w} + x^\perp$ and using $w^\top x^\perp = 0$ together with $w^\top \hat{w} = \lVert w \rVert$:

$$
y\,(\alpha\,\lVert w \rVert + b) \geq \gamma \;\;\Longleftrightarrow\;\; y\alpha \geq \frac{\gamma - yb}{\lVert w \rVert} \Longleftrightarrow 
    \begin{cases} 
        \alpha \geq -\frac{b}{\lVert w \rVert}+\frac{\gamma}{\lVert w \rVert} \quad y = 1 \\
        \alpha \leq -\frac{b}{\lVert w \rVert}-\frac{\gamma}{\lVert w \rVert} \quad y = -1
    \end{cases}
$$

The whole trick: **the constraint is purely about $\alpha$** (how far the point reaches along $\hat{w}$), and **$x^\perp$ is free** because it doesn't affect $w^\top x$ at all. A $d$-dimensional constraint becomes a 1D condition plus $d-1$ free directions.

### What each line of the sampler does

**Setting up the axis.**

```python
w_norm = w.norm()
w_hat = w / w_norm
```

Work in a coordinate system aligned with $w$. Everything that follows is measured along the unit vector $\hat{w}$ (perpendicular to the hyperplane) or in the $(d-1)$-dimensional subspace perpendicular to it (parallel to the hyperplane).

**Finding where the hyperplane sits on the $\hat{w}$-axis.**

```python
alpha_boundary = -bias / w_norm
```

The hyperplane $w^\top x + b = 0$ intersects the $\hat{w}$-axis where $\alpha \lVert w \rVert + b = 0$, i.e., at $\alpha = -b/\lVert w \rVert$. Points with this $\alpha$ sit exactly on the boundary.

**Converting functional margin to geometric margin.**

```python
geom_margin = margin / w_norm
```

The parameter `margin` is a functional margin ($\gamma$ in the formula). We want points at least $\gamma/\lVert w \rVert$ units of actual geometric distance away from the boundary along $\hat{w}$. Since $w^\top x + b$ measures distance times $\lVert w \rVert$, we divide by $\lVert w \rVert$ to get a real distance.

**Pushing $\alpha$ past the margin, on the right side.**

```python
offset = torch.empty(n).exponential_(1.0 / alpha_scale)
alpha = alpha_boundary + label * (geom_margin + offset)
```

`offset` is a random positive number (exponentially distributed — small values common, large values rare). Start at the boundary, step `geom_margin` units outward to clear the forbidden gap, then add `offset` more for natural spread. Multiplying by `label` ($\pm 1$) picks the side: positive points get pushed in the $+\hat{w}$ direction, negative points in the $-\hat{w}$ direction. Result: every positive point satisfies $w^\top x + b \geq +\gamma$ and every negative point satisfies $w^\top x + b \leq -\gamma$.

**Filling in the perpendicular directions freely.**

```python
g = torch.randn(n, d) * perp_scale
x_perp = g - (g @ w_hat).unsqueeze(1) * w_hat
```

Sample a random Gaussian `g`, then remove its component along $\hat{w}$. What's left lives in $w^\perp$ — it has no projection onto $\hat{w}$ and therefore doesn't affect whether the constraint is satisfied. The subtraction is the standard orthogonal-projection formula: take $g$, remove its shadow along $\hat{w}$, keep the perpendicular residual. `perp_scale` controls how spread out the data cloud looks along the boundary.

**Assembling the point.**

```python
return alpha.unsqueeze(1) * w_hat + x_perp
```

Put the two pieces back together: a carefully chosen step along $\hat{w}$ (guarantees the class constraint) plus a free random step perpendicular to it (gives natural-looking spread).

In [ ]:
# TODO: implement sample_data(n, w, bias, margin, label, alpha_scale=1.0, perp_scale=3.0)
#   Return a tensor of shape (n, d) whose rows satisfy
#       label * (w^T x + bias) >= margin
#   The hints in the markdown cell above walk you through each step.

In [ ]:
# TODO: set the dataset parameters used by the plot below.
#   Need: torch.manual_seed(...), bias (float), margin (float >= 0), w_star (2-D tensor).

In [ ]:
# TODO: generate the dataset.
#   Need: N (int), X (shape (N, 2)), Y (shape (N,) with values in {-1, +1}).
#   Use sample_data twice (once per class), concatenate, and label via X @ w_star + bias.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

# Split by class for coloring
pos = Y.squeeze() == 1
neg = Y.squeeze() == -1

ax.scatter(
    X[pos, 0],
    X[pos, 1],
    c="tab:blue",
    marker="o",
    s=60,
    label="y = +1",
    edgecolor="k",
)
ax.scatter(
    X[neg, 0],
    X[neg, 1],
    c="tab:orange",
    marker="s",
    s=60,
    label="y = -1",
    edgecolor="k",
)

# Decision boundary w^T x + b = 0 solved for x2:  x2 = -(w0*x1 + b) / w1
xs = np.linspace(X[:, 0].min().item(), X[:, 0].max().item(), 100)
ys = -(w_star[0] * xs + bias) / w_star[1]
ys_plus = -(w_star[0] * xs + bias - margin) / w_star[1]  # w^T x + b = +margin
ys_minus = -(w_star[0] * xs + bias + margin) / w_star[1]  # w^T x + b = -margin
ax.plot(xs, ys, "k--", lw=1.2, label=r"$w^\top x + b = 0$")
ax.plot(
    xs, ys_plus, "k:", lw=1.0, alpha=0.6, label=r"$w^\top x + b = \pm\,\mathrm{margin}$"
)
ax.plot(xs, ys_minus, "k:", lw=1.0, alpha=0.6)

ax.set_xlim(X[:, 0].min().item() - 1, X[:, 0].max().item() + 1)
ax.set_ylim(X[:, 1].min().item() - 1, X[:, 1].max().item() + 1)
ax.set_xlabel(r"$x_1$")
ax.set_ylabel(r"$x_2$")
ax.set_title(f"Random dataset (N={N}, margin={margin}, bias={bias})")
ax.axhline(0, color="gray", lw=0.5)
ax.axvline(0, color="gray", lw=0.5)
ax.set_aspect("equal")
ax.grid(alpha=0.3)
ax.legend(loc="best")

plt.tight_layout()
plt.show()

## The Perceptron Learning Algorithm

**Input**: ${\mathcal{D}=\{(x^{(i)}, y^{(i)})\}_{i=1}^N}$, $E=$ Number of epochs

Let $w_0=(\underbrace{0, \dots, 0}_{d\ \text{data dims}}, \underbrace{0}_{\text{bias}}) \in \mathbb{R}^{d+1}$, with one extra dimension for the bias (we need to remember to add to every input $x^{(i)}$ an extra 1 too)

```python
for e in range(E):
    for i in range(X.shape[0]):
        score = ...
        if score <= 0:
            w = ...
```

Remember, convergence is given by

$$\tau \leq \left(\frac{\sqrt{R^2 + 1}}{\tilde{\gamma}}\right)^2$$

where $\lVert x^{(i)} \rVert \leq R$, ${\tilde{\gamma} = \min_i \frac{y^{(i)}\tilde{w}^{*\top}\tilde{x}^{(i)}}{\lVert \tilde{w}^* \rVert}}$ is the normalized margin and $\tau$ is the number of misclassified steps

In [ ]:
# TODO: compute the Novikoff convergence bound.
#   Build X_bias (augment X with a column of 1s) and w_bias (augment w_star with the bias).
#   Compute:
#     R      = max_i ||x_i||               (max norm over the original X)
#     gamma  = empirical geometric margin  (min_i y_i * w_bias^T x_bias_i) / ||w_bias||
#     tau    = ((R^2 + 1)^(1/2) / gamma)^2 (the misclassification-step upper bound)
#   Return tau (and anything else you want to inspect).

In [ ]:
# TODO: implement the perceptron training loop.
#   Start w at zero in R^(d+1) (weights + bias). Use the augmented X_bias.
#   For each epoch, iterate over the samples; if y_i * (w^T x_i) <= 0, update
#       w <- w + y_i * x_i
#   Track total mistakes and stop early when an epoch makes none.
#   Produce: w (the learned weights incl. bias), mistakes (total count).

In [ ]:
# TODO: inspect the result — print or display w and the total mistake count.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

# Split by class for coloring
pos = Y.squeeze() == 1
neg = Y.squeeze() == -1

ax.scatter(
    X[pos, 0],
    X[pos, 1],
    c="tab:blue",
    marker="o",
    s=60,
    label="y = +1",
    edgecolor="k",
)
ax.scatter(
    X[neg, 0],
    X[neg, 1],
    c="tab:orange",
    marker="s",
    s=60,
    label="y = -1",
    edgecolor="k",
)

# True boundary w*^T x + b = 0 solved for x2:  x2 = -(w0*x1 + b) / w1
xs = np.linspace(X[:, 0].min().item(), X[:, 0].max().item(), 100)
ys = -(w_star[0] * xs + bias) / w_star[1]
ys_plus = -(w_star[0] * xs + bias - margin) / w_star[1]  # w*^T x + b = +margin
ys_minus = -(w_star[0] * xs + bias + margin) / w_star[1]  # w*^T x + b = -margin
ax.plot(xs, ys, "k--", lw=1.2, label=r"$w^{*\top} x + b = 0$")
ax.plot(
    xs,
    ys_plus,
    "k:",
    lw=1.0,
    alpha=0.6,
    label=r"$w^{*\top} x + b = \pm\,\mathrm{margin}$",
)
ax.plot(xs, ys_minus, "k:", lw=1.0, alpha=0.6)

# Found weight
xw, yw, bw = w.flatten().tolist()

x1 = np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 200)
if yw != 0:
    x2 = -(xw * x1 + bw) / yw
    ax.plot(x1, x2, color="red", lw=1.5, label=r"$\hat w^\top x + \hat b = 0$")
else:
    # Vertical line: xw * x1 + bw = 0  =>  x1 = -bw / xw
    ax.axvline(-bw / xw, color="red", lw=1.5, label=r"$\hat w^\top x + \hat b = 0$")

ax.set_xlim(X[:, 0].min().item() - 1, X[:, 0].max().item() + 1)
ax.set_ylim(X[:, 1].min().item() - 1, X[:, 1].max().item() + 1)
ax.set_xlabel(r"$x_1$")
ax.set_ylabel(r"$x_2$")
ax.set_title(f"Random dataset (N={N}, margin={margin}, bias={bias})")
ax.axhline(0, color="gray", lw=0.5)
ax.axvline(0, color="gray", lw=0.5)
ax.set_aspect("equal")
ax.grid(alpha=0.3)
ax.legend(loc="best")

plt.tight_layout()
plt.show()